# Embedding-Matrix Geometry Analysis — 4 models

Analyses and **compares the input token-embedding matrices**
(`model.model.embed_tokens.weight`) across four checkpoints of the SinLlama line:

| Key | Directory | Vocab | Role |
|-----|-----------|------:|------|
| `Llama-3-8B`        | `llama-3-8b`                  | 128,256 | original Meta base model |
| `SinLlama_v01`      | `SinLlama_v01`                | 139,336 | Sinhala model from Llama-3 (CPT, extended vocab) — **parent of cpt & instruct** |
| `SinLlama_cpt`      | `SinLlama_cpt`                | 139,336 | branched from v01 |
| `SinLlama_Instruct` | `SinLlama_Backtrianx_Instruct`| 139,336 | branched from v01 (Bactrian-X instruct) |

**Lineage:** `Llama-3-8B → SinLlama_v01 → {SinLlama_cpt, SinLlama_Instruct}`.
`cpt` and `instruct` are **parallel branches trained off v01** — they are *not*
sequential to each other. Comparisons are ordered to follow this hierarchy.

The three SinLlama models **extend the base vocabulary by 11,080 new tokens**
(ids ≥ 128256, almost all Sinhala). Because ids `0 … 128255` are the *same*
tokens in every model, we can directly track how each training stage moved the
**original (English) embeddings** and study the geometry of the **new Sinhala
embeddings**.

We load only the embedding tensor (~1 GB / model) out of shard 1 with
`safetensors` — never the full 8-B model. Follows `weights_analysis/todo.txt`.

> **Memory:** four embedding matrices + unit copies ≈ 18 GB RAM in float32.
> Fine on the MI300X pod / a 32 GB+ box. To run lighter, trim `MODEL_DIRS`.

## 0. Setup — imports, configuration, helpers

In [ ]:
# --- Core numerical / plotting stack -------------------------------------
import os, re, json, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- ML utilities ---------------------------------------------------------
import torch
from safetensors import safe_open
from transformers import AutoTokenizer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors
from scipy.cluster.hierarchy import linkage, dendrogram
import umap  # umap-learn

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

SEED = 42
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

# --- Portable model-path resolution (works on the laptop AND the pod) -----
CANDIDATE_ROOTS = [
    "/ml/SinLlama_CPT/models",
    os.path.expanduser("~/sinllama-continual-pretraining/models"),
    "/root/sinllama-continual-pretraining/models",
    "./models",
]
MODELS_ROOT = next((r for r in CANDIDATE_ROOTS if os.path.isdir(r)), CANDIDATE_ROOTS[0])

# logical name -> candidate directory names (case differs across machines).
# Order follows the model lineage: llama -> v01 -> {cpt, instruct}.
MODEL_DIRS = {
    "Llama-3-8B":        ["llama-3-8b"],
    "SinLlama_v01":      ["SinLlama_v01"],
    "SinLlama_cpt":      ["SinLlama_cpt"],
    "SinLlama_Instruct": ["SinLlama_Backtrianx_Instruct", "SinLlama_Backtrianx_instruct"],
}

def resolve_dir(root, candidates):
    "Case-insensitive match of a model directory under `root`."
    existing = {d.lower(): d for d in os.listdir(root)
                if os.path.isdir(os.path.join(root, d))}
    for c in candidates:
        if c.lower() in existing:
            return os.path.join(root, existing[c.lower()])
    raise FileNotFoundError(f"none of {candidates} under {root}")

MODEL_PATHS = {k: resolve_dir(MODELS_ROOT, v) for k, v in MODEL_DIRS.items()}
REF_MODEL = "Llama-3-8B"          # base model, used for single-model reference plots
ORIG_VOCAB = 128256               # size of the base Llama-3 vocabulary

FIG_DIR = os.path.join(os.path.dirname(MODELS_ROOT.rstrip("/")),
                       "weights_analysis", "figures")
if not os.path.isdir(os.path.dirname(FIG_DIR)):
    FIG_DIR = "figures"           # fallback: current directory
os.makedirs(FIG_DIR, exist_ok=True)

# Subset sizes for the expensive projection / clustering steps.
SUBSET_SIZE, TSNE_SIZE, HIER_SIZE = 8000, 3000, 1200
PAIRWISE_N, HUB_N = 5000, 5000

def savefig(name):
    "Save the current figure into FIG_DIR and also display it inline."
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, name), bbox_inches="tight")
    plt.show()

print("MODELS_ROOT:", MODELS_ROOT)
for k, p in MODEL_PATHS.items():
    print(f"  {k:20s} -> {p}")
print("Figures ->", FIG_DIR)

## 1. Load and inspect

Read only `model.embed_tokens.weight` from shard 1 of each checkpoint, cast
bf16 → float32 (numpy/sklearn need it), verify shapes and report dtype +
parameter count.

In [ ]:
def _embed_shard(model_path, weight="model.embed_tokens.weight"):
    "Locate the safetensors shard that stores the embedding matrix."
    idx = os.path.join(model_path, "model.safetensors.index.json")
    if os.path.exists(idx):
        weight_map = json.load(open(idx))["weight_map"]
        return os.path.join(model_path, weight_map[weight])
    return os.path.join(model_path, "model.safetensors")  # single-file fallback

def load_embedding_matrix(model_path, weight="model.embed_tokens.weight"):
    "Load ONLY the embedding tensor (not the whole model) and return float32 numpy."
    shard = _embed_shard(model_path, weight)
    with safe_open(shard, framework="pt", device="cpu") as f:
        w = f.get_tensor(weight)          # [vocab, hidden]
    return w.to(torch.float32).numpy(), w.dtype

# Load every embedding matrix + tokenizer into a registry dict.
MODELS = {}
for key, path in MODEL_PATHS.items():
    emb, dtype = load_embedding_matrix(path)
    tok = AutoTokenizer.from_pretrained(path)
    MODELS[key] = {"emb": emb, "dtype": dtype, "tok": tok, "path": path}
    print(f"{key:20s} shape={emb.shape}  dtype(orig)={dtype}  "
          f"params={emb.shape[0]*emb.shape[1]:,}")

SUMMARY = {k: {} for k in MODELS}          # scalar metrics for the final table

### 1b. Token categorisation

Tag every token id with a coarse category (English word, sub-word fragment,
punctuation, number, whitespace, special, **Sinhala**, other-non-English).
Sinhala is detected via the Unicode block `U+0D80–U+0DFF`; new tokens are those
with id ≥ 128256. Decoding ~139k tokens takes a few seconds per model.

In [ ]:
SIN_RE     = re.compile(r"[඀-෿]")               # Sinhala Unicode block
SPECIAL_RE = re.compile(r"^<\|.*\|>$")          # <|begin_of_text|> etc.

CAT_COLORS = {
    "english_word": "#1f77b4", "subword_fragment": "#aec7e8", "sinhala": "#d62728",
    "non_english_other": "#ff9896", "number": "#2ca02c", "punctuation": "#9467bd",
    "whitespace": "#8c564b", "special": "#7f7f7f", "other": "#bcbd22",
}

def classify(i, readable, piece, special_ids):
    "Map one token to a coarse category using its decoded string."
    if i in special_ids or SPECIAL_RE.match(piece or ""):
        return "special"
    s = readable.strip()
    if s == "":                       return "whitespace"
    if SIN_RE.search(readable):       return "sinhala"
    if not readable.isascii():        return "non_english_other"
    leading_space = readable[:1] == " "
    if s.isdigit():                   return "number"
    if all(not c.isalnum() for c in s):        return "punctuation"
    if s.replace("'", "").isalpha():
        return "english_word" if leading_space else "subword_fragment"
    if any(c.isdigit() for c in s):   return "number"
    return "other"

def build_categories(entry):
    "Per-token category array + human-readable strings for a model."
    tok = entry["tok"]; vocab = entry["emb"].shape[0]
    special_ids = set(tok.all_special_ids)
    pieces   = tok.convert_ids_to_tokens(list(range(vocab)))
    readable = [tok.decode([i]) for i in range(vocab)]
    cats = np.array([classify(i, readable[i], pieces[i], special_ids)
                     for i in range(vocab)])
    return cats, np.array(readable, dtype=object)

for key, entry in MODELS.items():
    cats, readable = build_categories(entry)
    entry["cats"] = cats; entry["readable"] = readable
    entry["is_new"] = np.arange(entry["emb"].shape[0]) >= ORIG_VOCAB
    print(f"\n=== {key} category counts ===")
    print(pd.Series(cats).value_counts().to_string())
    if entry["is_new"].any():
        print(f"new tokens (id>={ORIG_VOCAB}): {int(entry['is_new'].sum()):,}")

# Models that actually have the extended (Sinhala) vocabulary.
SINLLAMA = [k for k, e in MODELS.items() if e["is_new"].any()]
print("\nSinLlama variants:", SINLLAMA)

### 1c. Unit-normalised embeddings & norms

Precompute per-token L2 norms and a unit-normalised copy (guarding against
all-zero reserved tokens) — reused by every cosine/anisotropy step.

In [ ]:
for key, entry in MODELS.items():
    emb = entry["emb"]
    norms = np.linalg.norm(emb, axis=1)
    safe = np.where(norms == 0, 1.0, norms)
    entry["norms"] = norms
    entry["unit"]  = (emb / safe[:, None]).astype(np.float32)
    print(f"{key:20s} norm mean={norms.mean():.3f}  std={norms.std():.3f}  "
          f"zero-vectors={int((norms==0).sum())}")

### 1d. Embedding drift from base — original (shared) tokens

Ids `0 … 128255` are the same token in all four models, so per-token cosine
similarity between the base model and each SinLlama stage measures **how far
continual pretraining moved each original embedding**. cos ≈ 1 → barely moved;
lower → the token was re-shaped by Sinhala training.

In [ ]:
base_unit = MODELS[REF_MODEL]["unit"][:ORIG_VOCAB]
plt.figure(figsize=(8.5, 4.5))
drift_rows = []
for key in SINLLAMA:
    v = MODELS[key]["unit"][:ORIG_VOCAB]
    cos = (base_unit * v).sum(1)                      # per-token cosine to base
    SUMMARY[key]["mean_orig_drift_cos"] = float(cos.mean())
    plt.hist(cos, bins=150, alpha=0.5, density=True, label=f"{key} (mean={cos.mean():.3f})")
    # most-moved original tokens
    moved = np.argsort(cos)[:8]
    drift_rows.append({"model": key, "mean_cos_to_base": round(float(cos.mean()), 4),
                       "frac_cos<0.9": round(float((cos < 0.9).mean()), 4),
                       "most_moved": [MODELS[key]["readable"][i] for i in moved]})
plt.title("Per-token cosine similarity to base Llama-3 (original tokens only)")
plt.xlabel("cos(base, variant)"); plt.ylabel("density"); plt.legend(fontsize=8)
savefig("01d_original_token_drift.png")
display(pd.DataFrame(drift_rows).set_index("model"))

## 2. Basic statistics

Element-wise mean/std/min/max and the per-token L2-norm distribution.

In [ ]:
rows = []
for key, entry in MODELS.items():
    e = entry["emb"]
    rows.append({"model": key, "shape": str(e.shape), "mean": e.mean(),
                 "std": e.std(), "min": e.min(), "max": e.max(),
                 "norm_mean": entry["norms"].mean(), "norm_std": entry["norms"].std()})
display(pd.DataFrame(rows).set_index("model").round(4))

In [ ]:
# Per-token norm distribution (overlaid) + sampled value histogram.
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for key, entry in MODELS.items():
    ax[0].hist(entry["norms"], bins=120, alpha=0.5, label=key, density=True)
ax[0].set_title("Distribution of embedding L2 norms")
ax[0].set_xlabel("||embedding||"); ax[0].set_ylabel("density"); ax[0].legend(fontsize=8)
for key, entry in MODELS.items():
    idx = rng.choice(entry["emb"].shape[0], size=300, replace=False)
    ax[1].hist(entry["emb"][idx].ravel(), bins=200, alpha=0.5, label=key, density=True)
ax[1].set_title("Distribution of embedding weight values (sampled)")
ax[1].set_xlabel("weight value"); ax[1].set_ylabel("density"); ax[1].legend(fontsize=8)
savefig("02_basic_distributions.png")

In [ ]:
# Outlier check: tokens whose norm is >3 std from the mean norm.
for key, entry in MODELS.items():
    n = entry["norms"]
    out = np.where((n > n.mean() + 3*n.std()) | (n < n.mean() - 3*n.std()))[0]
    top = out[np.argsort(-n[out])][:8]
    print(f"{key}: {len(out)} norm-outliers (>|3 sigma|). "
          f"largest: {[(repr(entry['readable'][i]), round(float(n[i]),2)) for i in top]}")

## 3. Cosine-similarity analysis

Cosine of selected token pairs and top-k nearest neighbours per model.

In [ ]:
def single_token_id(tok, s):
    "Return the id if `s` (optionally with a leading space) is ONE token, else None."
    for cand in (s, " " + s):
        ids = tok.encode(cand, add_special_tokens=False)
        if len(ids) == 1:
            return ids[0]
    return None

def nearest_neighbors(entry, token_id, k=10):
    "Top-k cosine neighbours of a token id within the same model."
    sims = entry["unit"] @ entry["unit"][token_id]
    order = np.argsort(-sims); order = order[order != token_id][:k]
    return [(int(j), entry["readable"][j], float(sims[j])) for j in order]

PAIRS = [("king", "queen"), ("cat", "dog"), ("run", "running"),
         ("good", "bad"), ("one", "two")]
for key, entry in MODELS.items():
    print(f"\n=== {key}: cosine similarity of selected pairs ===")
    for a, b in PAIRS:
        ia, ib = single_token_id(entry["tok"], a), single_token_id(entry["tok"], b)
        if ia is None or ib is None:
            print(f"  {a!r:>10} ~ {b!r:<10}  (not single tokens)"); continue
        print(f"  {a!r:>10} ~ {b!r:<10}  cos = {float(entry['unit'][ia] @ entry['unit'][ib]):+.3f}")

In [ ]:
# Top-k nearest neighbours for a spread of token TYPES.
PROBES = ["cat", "dog", "computer", "run", "the", "3", ".", "!"]
for key, entry in MODELS.items():
    print(f"\n################  {key}  ################")
    for w in PROBES:
        tid = single_token_id(entry["tok"], w)
        if tid is None:
            print(f"[{w!r}] not a single token"); continue
        nn = nearest_neighbors(entry, tid, k=8)
        print(f"[{w!r:>10}] -> " + ", ".join(f"{r!r}({s:.2f})" for _, r, s in nn))

In [ ]:
# Similarity heatmap for a small mixed token group (reference model).
GROUP = ["one", "two", "three", "cat", "dog", "king", "queen", ".", ",", "!"]
entry = MODELS[REF_MODEL]
ids, labels = [], []
for w in GROUP:
    tid = single_token_id(entry["tok"], w)
    if tid is not None:
        ids.append(tid); labels.append(w)
sub = entry["unit"][ids]; sim = sub @ sub.T
plt.figure(figsize=(7, 6))
sns.heatmap(sim, xticklabels=labels, yticklabels=labels, cmap="viridis",
            annot=True, fmt=".2f", square=True, cbar_kws={"label": "cosine"})
plt.title(f"Token-group cosine-similarity matrix ({REF_MODEL})")
savefig("03_similarity_heatmap.png")

## 4. Dimensionality reduction (PCA / UMAP / t-SNE)

Project a random subset to 2-D, coloured by token category, per model.

In [ ]:
def sample_indices(entry, n):
    v = entry["emb"].shape[0]
    return rng.choice(v, size=min(n, v), replace=False)

def scatter_by_category(coords, cats, title, fname):
    plt.figure(figsize=(9, 7))
    for c in [x for x in CAT_COLORS if x in set(cats)]:
        m = cats == c
        plt.scatter(coords[m, 0], coords[m, 1], s=6, alpha=0.5,
                    c=CAT_COLORS[c], label=f"{c} ({m.sum()})")
    plt.legend(markerscale=2, fontsize=8, loc="best")
    plt.title(title); plt.xlabel("dim-1"); plt.ylabel("dim-2")
    savefig(fname)

# Full-vocab PCA per model (explained variance) + 2-D scatter on a subset.
for key, entry in MODELS.items():
    pca = PCA(n_components=50, svd_solver="randomized", random_state=SEED).fit(entry["emb"])
    entry["pca"] = pca
    SUMMARY[key]["pc1_var_ratio"] = float(pca.explained_variance_ratio_[0])
    SUMMARY[key]["var_top10"] = float(pca.explained_variance_ratio_[:10].sum())
    print(f"{key}: PC1 {pca.explained_variance_ratio_[0]*100:.1f}% | "
          f"top-10 {pca.explained_variance_ratio_[:10].sum()*100:.1f}%")
    idx = sample_indices(entry, SUBSET_SIZE); entry["subset_idx"] = idx
    coords = entry["pca"].transform(entry["emb"][idx])[:, :2]
    scatter_by_category(coords, entry["cats"][idx],
                        f"PCA (first 2 PCs) — {key}", f"04_pca_{key}.png")

In [ ]:
# Cumulative explained variance, all models overlaid.
plt.figure(figsize=(8, 4.5))
for key, entry in MODELS.items():
    ev = entry["pca"].explained_variance_ratio_
    plt.plot(np.arange(1, len(ev)+1), np.cumsum(ev), marker="o", ms=3, label=key)
plt.xlabel("number of principal components"); plt.ylabel("cumulative explained variance")
plt.title("PCA cumulative explained variance (top 50 PCs)"); plt.legend()
savefig("04_pca_cumvar.png")

In [ ]:
# UMAP on the same subset (non-linear manifold view).
for key, entry in MODELS.items():
    coords = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine",
                       random_state=SEED).fit_transform(entry["emb"][entry["subset_idx"]])
    scatter_by_category(coords, entry["cats"][entry["subset_idx"]],
                        f"UMAP (cosine) — {key}", f"04_umap_{key}.png")

In [ ]:
# t-SNE on a smaller subset (quadratic cost).
for key, entry in MODELS.items():
    idx = sample_indices(entry, TSNE_SIZE)
    coords = TSNE(n_components=2, perplexity=30, init="pca",
                  learning_rate="auto", random_state=SEED).fit_transform(entry["emb"][idx])
    scatter_by_category(coords, entry["cats"][idx], f"t-SNE — {key}", f"04_tsne_{key}.png")

## 5. Clustering

K-Means (k=12) + Ward hierarchical clustering, cross-tabulated against token
category.

In [ ]:
K = 12
for key, entry in MODELS.items():
    idx = entry["subset_idx"]; X = entry["emb"][idx]
    km = KMeans(n_clusters=K, n_init=10, random_state=SEED).fit(X)
    coords = entry["pca"].transform(X)[:, :2]
    plt.figure(figsize=(8, 6.5))
    plt.scatter(coords[:, 0], coords[:, 1], s=6, alpha=0.5, c=km.labels_, cmap="tab20")
    plt.title(f"K-Means (k={K}) in PCA space — {key}"); plt.xlabel("PC1"); plt.ylabel("PC2")
    savefig(f"05_kmeans_{key}.png")
    print(f"\n=== {key}: cluster x category (rows=cluster) ===")
    display(pd.crosstab(km.labels_, entry["cats"][idx]))

In [ ]:
# Ward hierarchical clustering dendrogram (reference model).
entry = MODELS[REF_MODEL]; idx = sample_indices(entry, HIER_SIZE)
Z = linkage(entry["emb"][idx], method="ward")
plt.figure(figsize=(11, 4))
dendrogram(Z, no_labels=True, color_threshold=None)
plt.title(f"Ward hierarchical clustering dendrogram — {REF_MODEL} (n={len(idx)})")
plt.xlabel("tokens"); plt.ylabel("merge distance")
savefig("05_dendrogram.png")

## 6. Spectral analysis (SVD / covariance spectrum)

Covariance eigen-spectrum + effective rank (`exp(entropy)` and participation
ratio). Low effective rank vs 4096 → representations live in a small subspace.

In [ ]:
def spectrum_metrics(emb):
    Xc = emb - emb.mean(axis=0, keepdims=True)
    cov = (Xc.T @ Xc) / (Xc.shape[0] - 1)
    eig = np.clip(np.linalg.eigvalsh(cov)[::-1], 0, None)
    p = eig / eig.sum(); p = p[p > 0]
    return eig, float(np.exp(-(p*np.log(p)).sum())), float((eig.sum()**2)/np.sum(eig**2))

plt.figure(figsize=(12, 4.5))
for key, entry in MODELS.items():
    eig, er, pr = spectrum_metrics(entry["emb"])
    SUMMARY[key]["eff_rank_entropy"] = er; SUMMARY[key]["participation_ratio"] = pr
    print(f"{key}: effective rank (exp-H)={er:.1f} | participation ratio={pr:.1f}")
    plt.subplot(1, 2, 1); plt.semilogy(eig, label=key)
    plt.subplot(1, 2, 2); plt.plot(np.cumsum(eig)/eig.sum(), label=key)
plt.subplot(1, 2, 1); plt.title("Covariance eigenvalue spectrum (log)")
plt.xlabel("component"); plt.ylabel("eigenvalue"); plt.legend()
plt.subplot(1, 2, 2); plt.title("Cumulative explained variance")
plt.xlabel("component"); plt.ylabel("fraction"); plt.legend()
savefig("06_spectrum.png")

## 7. Anisotropy analysis

Mean cosine-to-mean and the Mu et al. (2018) isotropy score
`I = min_c Z(c)/max_c Z(c)` (I→1 isotropic, I→0 anisotropic).

In [ ]:
def isotropy_score(emb, pca, n_axes=15, sample=20000):
    idx = rng.choice(emb.shape[0], size=min(sample, emb.shape[0]), replace=False)
    axes = pca.components_[:n_axes]
    Z = np.exp(emb[idx] @ axes.T).sum(axis=0)
    return float(Z.min() / Z.max())

for key, entry in MODELS.items():
    mean_vec = entry["emb"].mean(axis=0); mean_unit = mean_vec / np.linalg.norm(mean_vec)
    cos_to_mean = entry["unit"] @ mean_unit
    iso = isotropy_score(entry["emb"], entry["pca"])
    SUMMARY[key]["avg_cos_to_mean"] = float(cos_to_mean.mean())
    SUMMARY[key]["isotropy_score"] = iso
    print(f"{key}: mean cos-to-mean = {cos_to_mean.mean():+.3f} "
          f"(|mean vec|={np.linalg.norm(mean_vec):.3f}) | isotropy I = {iso:.4f}")

plt.figure(figsize=(8, 4.5))
for key, entry in MODELS.items():
    mv = entry["emb"].mean(axis=0); mv /= np.linalg.norm(mv)
    plt.hist(entry["unit"] @ mv, bins=120, alpha=0.5, density=True, label=key)
plt.axvline(0, color="k", lw=1, ls="--")
plt.title("Cosine similarity of each token embedding to the mean vector")
plt.xlabel("cos(embedding, mean)"); plt.ylabel("density"); plt.legend(fontsize=8)
savefig("07_anisotropy.png")

## 8. Embedding-norm analysis

Mean norm per token category, and largest/smallest-norm tokens, per model.

In [ ]:
for key, entry in MODELS.items():
    df = pd.DataFrame({"cat": entry["cats"], "norm": entry["norms"]})
    order = df.groupby("cat")["norm"].mean().sort_values(ascending=False)
    plt.figure(figsize=(9, 4))
    sns.barplot(x=order.index, y=order.values,
                palette=[CAT_COLORS.get(c, "#333") for c in order.index])
    plt.xticks(rotation=35, ha="right"); plt.ylabel("mean L2 norm")
    plt.title(f"Mean embedding norm by category — {key}")
    savefig(f"08_norm_by_category_{key}.png")
    n = entry["norms"]
    print(f"{key} largest-norm:",
          [(repr(entry['readable'][i]), round(float(n[i]),2)) for i in np.argsort(-n)[:8]])

## 9. Token-category analysis

Per-category count, mean norm and mean intra-category cosine (tightness).

In [ ]:
def category_report(entry, max_per_cat=2000):
    rows = []
    for c in sorted(set(entry["cats"])):
        ids = np.where(entry["cats"] == c)[0]
        if len(ids) > max_per_cat:
            ids = rng.choice(ids, size=max_per_cat, replace=False)
        u = entry["unit"][ids]; sims = u @ u.T
        iu = np.triu_indices(len(ids), k=1)
        mean_cos = float(sims[iu].mean()) if len(iu[0]) else float("nan")
        rows.append({"category": c, "count": int((entry["cats"] == c).sum()),
                     "mean_norm": float(entry["norms"][entry["cats"] == c].mean()),
                     "mean_intra_cos": mean_cos})
    return pd.DataFrame(rows).set_index("category")

for key, entry in MODELS.items():
    print(f"\n=== {key} category report ===")
    display(category_report(entry).round(4))

In [ ]:
# New (Sinhala) vs original token norms, compared ACROSS the SinLlama variants.
plt.figure(figsize=(9, 4.6))
rows = []
for key in SINLLAMA:
    entry = MODELS[key]; is_new = entry["is_new"]
    new_norm, old_norm = entry["norms"][is_new], entry["norms"][~is_new]
    plt.hist(new_norm, bins=120, alpha=0.5, density=True,
             label=f"{key} new (mean={new_norm.mean():.2f})")
    rows.append({"model": key, "orig_mean_norm": round(float(old_norm.mean()), 3),
                 "new_mean_norm": round(float(new_norm.mean()), 3),
                 "new/orig": round(float(new_norm.mean()/old_norm.mean()), 3)})
plt.title("New (Sinhala) token norm distribution across SinLlama stages")
plt.xlabel("||embedding||"); plt.ylabel("density"); plt.legend(fontsize=8)
savefig("09_new_token_norms.png")
display(pd.DataFrame(rows).set_index("model"))

## 10. Neighbourhood analysis

Nearest neighbours of probe tokens; and, for the SinLlama models, neighbours of
the **same Sinhala tokens** across stages (same tokenizer → same ids).

In [ ]:
PROBE_WORDS = ["cat", "dog", "computer", "run", "happy", "3", "."]
for key, entry in MODELS.items():
    print(f"\n################  {key}  ################")
    for w in PROBE_WORDS:
        tid = single_token_id(entry["tok"], w)
        if tid is None:
            print(f"[{w!r}] not a single token"); continue
        nn = nearest_neighbors(entry, tid, k=10)
        print(f"[{w!r:>10}] -> " + ", ".join(f"{r!r}({s:.2f})" for _, r, s in nn))

In [ ]:
# Same Sinhala tokens, neighbours compared across SinLlama variants.
ref = MODELS[SINLLAMA[0]]
new_ids = np.where(ref["is_new"])[0]
sin_probes = [int(i) for i in new_ids
              if len(ref["readable"][i].strip()) >= 2][:5]
for tid in sin_probes:
    print(f"\n=== Sinhala token {ref['readable'][tid]!r} (id {tid}) ===")
    for key in SINLLAMA:
        nn = nearest_neighbors(MODELS[key], tid, k=6)
        print(f"  {key:20s} -> " + ", ".join(f"{r!r}({s:.2f})" for _, r, s in nn))

## 11. Frequency-based analysis (optional)

Enable by dropping a `{token_id: count}` JSON at `FREQ_PATH`.

In [ ]:
FREQ_PATH = os.path.join(FIG_DIR, "..", "token_freq.json")
if os.path.exists(FREQ_PATH):
    freq_map = {int(k): v for k, v in json.load(open(FREQ_PATH)).items()}
    for key, entry in MODELS.items():
        freq = np.array([freq_map.get(i, 0) for i in range(entry["emb"].shape[0])], float)
        mask = freq > 0
        plt.figure(figsize=(7, 5))
        plt.scatter(np.log1p(freq[mask]), entry["norms"][mask], s=4, alpha=0.3)
        plt.xlabel("log(1+frequency)"); plt.ylabel("embedding norm")
        plt.title(f"Frequency vs norm — {key}"); savefig(f"11_freq_vs_norm_{key}.png")
        print(f"{key}: corr = {np.corrcoef(np.log1p(freq[mask]), entry['norms'][mask])[0,1]:.3f}")
else:
    print(f"No frequency file at {FREQ_PATH} -> skipping section 11.")

## 12. Distance distribution & hubness

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for key, entry in MODELS.items():
    idx = sample_indices(entry, PAIRWISE_N); u = entry["unit"][idx]
    sims = (u @ u.T)[np.triu_indices(len(idx), k=1)]
    SUMMARY[key]["mean_pairwise_cos"] = float(sims.mean())
    SUMMARY[key]["var_pairwise_cos"]  = float(sims.var())
    ax[0].hist(sims, bins=150, alpha=0.5, density=True, label=key)
    print(f"{key}: mean pairwise cos = {sims.mean():+.3f}, var = {sims.var():.4f}")
    hidx = sample_indices(entry, HUB_N)
    nn = NearestNeighbors(n_neighbors=11, metric="cosine").fit(entry["unit"][hidx])
    _, neigh = nn.kneighbors(entry["unit"][hidx])
    kocc = np.bincount(neigh[:, 1:].ravel(), minlength=len(hidx))
    ax[1].hist(kocc, bins=range(0, kocc.max()+2), alpha=0.5, density=True, label=key)
    print(f"   top hubs: {[(repr(entry['readable'][hidx[h]]), int(kocc[h])) for h in np.argsort(-kocc)[:5]]}")
ax[0].set_title("Pairwise cosine-similarity distribution (sample)")
ax[0].set_xlabel("cosine"); ax[0].set_ylabel("density"); ax[0].legend(fontsize=8)
ax[1].set_title("Hubness: k-occurrence distribution (k=10)")
ax[1].set_xlabel("# times a token is in others' NN list"); ax[1].legend(fontsize=8)
savefig("12_distance_hubness.png")

## 13. Visualisation index

In [ ]:
print("Figures written to", FIG_DIR, ":")
for f in sorted(os.listdir(FIG_DIR)):
    print("  -", f)

## 14. Research questions — summary table

Scalar metrics for all four models, then programmatic answers.

In [ ]:
summary_df = pd.DataFrame(SUMMARY).T
display(summary_df.round(4))

d = SUMMARY
def cmp(metric, fmt="{:.3f}"):
    return " | ".join(f"{k}: {fmt.format(d[k][metric])}" for k in d if metric in d[k])

print("\n================ ANSWERS ================")
print(f"1. Anisotropic? mean cos-to-mean -> {cmp('avg_cos_to_mean')}")
print(f"   isotropy I (1=isotropic) -> {cmp('isotropy_score','{:.4f}')}")
print(f"2. Uniformly distributed? mean pairwise cosine -> {cmp('mean_pairwise_cos')}")
print(f"3/4. Semantic / category clustering -> see PCA/UMAP/t-SNE + KMeans cross-tabs.")
print(f"5. Hub tokens? -> see k-occurrence tails (section 12).")
print(f"6. PC1 variance -> {cmp('pc1_var_ratio')}; top-10 -> {cmp('var_top10')}")
print(f"7. Effective rank (exp-H) -> {cmp('eff_rank_entropy','{:.1f}')} of 4096.")
print(f"8. Original-token drift from base -> {cmp('mean_orig_drift_cos','{:.4f}')}")

### Reading the 4-model comparison

* **Original-token drift (§1d)** is the clearest training signal (all measured
  against base Llama-3): `SinLlama_v01` drifts from base, then its two sibling
  branches `SinLlama_cpt` and `SinLlama_Instruct` drift further — compare the two
  siblings against their shared parent v01.
* **New-token geometry (§9)** — do the 11,080 Sinhala embeddings gain norm /
  structure across v01 and its cpt/instruct branches? Small norms = under-trained.
* **Effective rank & anisotropy (§6/§7)** — did training expand or collapse the
  intrinsic dimensionality of the embedding space?
* **Sinhala neighbours (§10)** — does each model build a more coherent Sinhala
  sub-space (Sinhala tokens finding Sinhala neighbours)?